# U7-03 · Q-learning y SARSA con Gymnasium

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 7, apartados 7.5 y 7.6*

Ejecuta cada celda con **Mayús + Intro**. La primera celda instala Gymnasium en el navegador (tarda unos segundos).

In [ ]:
%pip install -q gymnasium
import gymnasium
print("Gymnasium", gymnasium.__version__)

## 1. El acantilado (ejemplo 4)

In [ ]:
import numpy as np
import gymnasium as gym

def entrenar(metodo, episodios=500, alfa=0.5, eps=0.1, semilla=0):
    env = gym.make("CliffWalking-v1")
    rng = np.random.default_rng(semilla)
    Q = np.zeros((48, 4))
    def elegir(s):                         # política épsilon-voraz
        if rng.random() < eps:
            return int(rng.integers(4))
        return int(np.argmax(Q[s]))
    retornos = []
    for ep in range(episodios):
        s, _ = env.reset(seed=semilla * 10_000 + ep)
        a, G, fin = elegir(s), 0, False
        while not fin:
            s2, r, term, trunc, _ = env.step(a)
            a2 = elegir(s2)
            if metodo == "q":
                siguiente = np.max(Q[s2])      # Q-learning
            else:
                siguiente = Q[s2, a2]          # SARSA
            Q[s, a] += alfa * (r + siguiente * (not term) - Q[s, a])
            s, a, G, fin = s2, a2, G + r, term or trunc
        retornos.append(G)
    return Q, np.array(retornos)

In [ ]:
import matplotlib.pyplot as plt
curvas = {}
for m in ["q", "sarsa"]:
    R = np.array([entrenar(m, semilla=i)[1] for i in range(10)])
    curvas[m] = R.mean(axis=0)
    print(m, "retorno medio, últimos 100 episodios:", R[:, -100:].mean().round(1))
for m, nombre in [("q", "Q-learning"), ("sarsa", "SARSA")]:
    plt.plot(np.convolve(curvas[m], np.ones(10) / 10, mode="valid"), label=nombre)
plt.ylim(-120, 0); plt.xlabel("episodio"); plt.ylabel("retorno"); plt.legend(); plt.show()

El libro usa 50 semillas; aquí 10 para que tarde menos. Ahora dibujamos el camino voraz aprendido con la semilla 1:

In [ ]:
def camino(Q):
    env = gym.make("CliffWalking-v1")
    s, _ = env.reset(seed=0)
    ruta = [s]
    for _ in range(60):
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        ruta.append(s)
        if term or trunc:
            break
    return ruta

for m in ["q", "sarsa"]:
    ruta = camino(entrenar(m, semilla=1)[0])
    rejilla = np.full((4, 12), ".")
    rejilla[3, 1:11] = "C"                     # acantilado
    for s in ruta:
        rejilla[s // 12, s % 12] = "*"
    print(m, f"({len(ruta) - 1} pasos)")
    print("\n".join(" ".join(f) for f in rejilla), "\n")

## 2. El lago helado (ejemplo 5)

In [ ]:
import numpy as np
import gymnasium as gym

def crear():
    return gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)

def q_learning(episodios=20_000, alfa=0.1, gamma=0.99, semilla=0):
    env, rng = crear(), np.random.default_rng(semilla)
    Q = np.zeros((16, 4))
    for ep in range(episodios):
        eps = max(0.05, 1 - ep / (0.5 * episodios))   # explora menos
        s, _ = env.reset(seed=semilla * 100_000 + ep)
        fin = False
        while not fin:
            if rng.random() < eps:
                a = int(rng.integers(4))
            else:
                a = int(np.argmax(Q[s]))
            s2, r, term, trunc, _ = env.step(a)
            Q[s, a] += alfa * (r + gamma * np.max(Q[s2]) * (not term)
                               - Q[s, a])
            s, fin = s2, term or trunc
    return Q

def tasa_exito(politica, n=10_000):
    env, exitos = crear(), 0
    for i in range(n):
        s, _ = env.reset(seed=123 + i)
        fin = False
        while not fin:
            s, r, term, trunc, _ = env.step(int(politica[s]))
            fin = term or trunc
        exitos += r
    return exitos / n

In [ ]:
Q = q_learning(20_000, semilla=0)
print("Tasa de éxito:", tasa_exito(Q.argmax(axis=1), n=2000))
flechas = np.array(list("←↓→↑"))[Q.argmax(axis=1)].reshape(4, 4)
mapa = np.array([list(f) for f in crear().unwrapped.desc.astype(str)])
print(np.where(np.isin(mapa, ["H", "G"]), mapa, flechas))

**Prueba.** Entrena con 500 y con 2000 episodios y compara la tasa de éxito. Cambia la semilla: ¿es igual de estable?

> Watkins, C. J. C. H. y Dayan, P. (1992). Q-learning. *Machine Learning*, 8, 279–292. https://doi.org/10.1007/BF00992698
>
> Towers, M. et al. (2024). Gymnasium: A Standard Interface for Reinforcement Learning Environments. https://doi.org/10.48550/arXiv.2407.17032